# Topic Modelling: IMDB Movie Review

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import libraries
import os
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import pandas as pd
import numpy as np

In [ ]:
ROOT = '/content/drive/MyDrive/IMDB_Data/aclImdb/train/pos'

In [ ]:
# extract positive reviews
reviews = []
files = os.listdir(ROOT)
for file in tqdm(files):
    path = os.path.join(ROOT, file)
    if os.path.isfile(path):
        with open(path, 'r') as fin:
            reviews.append(fin.read())

100%|██████████| 12500/12500 [05:09<00:00, 40.45it/s] 


In [ ]:
print(reviews[0])

Full House is a great family show. However, after watching some episodes over and over again I've realized that they're incredibly boring and they seem to shelter themselves from the outside world a lot. Yes, there is a lot of comedy, but there are times when it's incredibly cheesy. It's not like I hate it, but just don't watch them over and over again because they get old quick. Probably the best season is the first.<br /><br />Full House is about widower Danny Tanner(Bob Saget)and his three daughters D.J. (Candace Cameron) Stephanie (Jodie Sweetin) and Michelle (Mary-Kate and Ashley). When Danny's wife dies the he is in need of some help. So, his best friend Joey (Dave Coulier) and the girls' Uncle Jesse (John Stamos) moves in with them. Once they live there together they find they can't live without each other. <br /><br />Full House reminds you just how important family is and that you can always go home again.


## Feature Extraction

In [ ]:
# convert data to tfidf model
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(reviews)   # Document-term matrix

# visualize the tfidf data
pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())

,00,000,000s,003830,006,007,0079,0080,0083,0093638,...,élan,émigré,émigrés,était,état,étc,êxtase,ís,østbye,über
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12495,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12496,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12497,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12498,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Non-Negative Matrix Factorization(NMF) Decomposition

In [ ]:
N_TOPICS = 15
nmf = NMF(n_components=N_TOPICS, max_iter=1000)
W = nmf.fit_transform(X)  # Document-topic matrix
H = nmf.components_       # Topic-term matrix

In [ ]:
# Top 10 words per topic

words = np.array(vectorizer.get_feature_names_out())
topic_words = pd.DataFrame(np.zeros((N_TOPICS, 10)), index=[f'Topic {i + 1}' for i in range(N_TOPICS)],
                           columns=[f'Word {i + 1}' for i in range(10)]).astype(str)
for i in range(N_TOPICS):
    ix = H[i].argsort()[::-1][:10]
    topic_words.iloc[i] = words[ix]

topic_words

,Word 1,Word 2,Word 3,Word 4,Word 5,Word 6,Word 7,Word 8,Word 9,Word 10
Topic 1,br,10,ll,spoilers,end,just,simply,yes,plot,spoiler
Topic 2,movie,movies,watch,recommend,saw,10,definitely,makes,enjoyed,feel
Topic 3,film,films,scenes,director,plot,festival,cinema,work,characters,art
Topic 4,series,episode,episodes,season,tv,trek,characters,seasons,shows,television
Topic 5,man,role,character,performance,john,plays,best,played,actor,does
Topic 6,like,really,think,just,don,know,people,say,didn,did
Topic 7,life,story,people,world,real,war,characters,way,lives,true
Topic 8,funny,comedy,laugh,hilarious,fun,jokes,eddie,humor,funniest,murphy
Topic 9,good,story,action,pretty,bad,acting,really,plot,nice,scenes
Topic 10,family,kids,old,children,disney,years,father,little,son,young


In [ ]:
# Create a topic mapping

topic_mapping = {
    'Topic 4': 'TV',
    'Topic 7': 'War',
    'Topic 8': 'Comedy',
    'Topic 10': 'Music',
    'Topic 12': 'Book Adaptation',
    'Topic 13': 'Horror'
}

In [ ]:
# decorate the W (document-topic matrix)
result = pd.DataFrame(W, columns=[f'Topic {i+1}' for i in range(0, N_TOPICS)])
result.head()

,Topic 1,Topic 2,Topic 3,Topic 4,Topic 5,Topic 6,Topic 7,Topic 8,Topic 9,Topic 10,Topic 11,Topic 12,Topic 13,Topic 14,Topic 15
0,0.027673,0.000000,0.000000,0.017523,0.000000,0.017725,0.000000,0.012080,0.000000,0.056049,0.000000,0.000000,0.011372,0.015890,0.017013
1,0.010482,0.008872,0.000000,0.000651,0.022566,0.004989,0.000000,0.006546,0.004387,0.000000,0.000000,0.000000,0.018410,0.040005,0.016959
2,0.028277,0.029831,0.037178,0.009877,0.000000,0.016617,0.015298,0.000000,0.000000,0.023759,0.000000,0.002444,0.000000,0.007868,0.003713
3,0.000000,0.042531,0.066385,0.000000,0.000000,0.002212,0.066096,0.000000,0.008323,0.000000,0.000108,0.000000,0.000000,0.046118,0.000000
4,0.007575,0.000000,0.065916,0.000000,0.000000,0.012528,0.003676,0.000000,0.006017,0.000000,0.004988,0.003052,0.114084,0.018814,0.000000


In [ ]:
# identify the topic with maximum score of topic
result['max_topic'] = result.apply(lambda x: topic_mapping.get(x.idxmax()), axis=1)

In [ ]:
# print top 10 topics
result[pd.notnull(result['max_topic'])].head(10)['max_topic']

,max_topic
0,Music
4,Horror
9,TV
10,TV
13,Horror
16,Horror
17,Horror
18,TV
20,Horror
21,Horror


In [ ]:
# check for review 4, horror movie
reviews[4]

"Italy produced a lot of really great and original horror films in the 1960's - and this is certainly one of them! The first thing you will notice about Danse Macabre is the style of the film. Shot in beautiful black and white, and due to director Antonio Margheriti's use of lighting; the film almost looks like it could be a German expressionistic horror film. This, coupled with the horror-filled plot line ensures that Danse Macabre is a film that truly captures the essence of horror. Of course, the fact that the beautiful Barbara Steele appears in the film doesn't harm matters - and the good news continues as, in this film, she gets to flex her acting muscles more than she did in the films that made her famous. The plot is very aware of the time in which this was released, and so incorporates the great Edgar Allen Poe. We follow Alan Foster, a writer who accepts a bet from Poe himself and Lord Blackwood that he can't spend an entire night in the latter's creepy old castle. Everyone th